In [1]:
import pickle

with open('data/model.pkl', 'rb') as file:
    model = pickle.load(file)

print(model)

{'a': 5, 'b': 13}


In [2]:
import numpy as np  # на всякий случай, если нужно

features = [1, 1, 1, 0.661212487096872]
prediction = model.predict([features])[0]  # или np.array(features).reshape(1, -1)

rounded_prediction = round(prediction, 3)
print(rounded_prediction)

AttributeError: 'dict' object has no attribute 'predict'

In [ ]:
# предполагаем, что модель уже загружена в переменную model
dict_ab = {'a': model.a, 'b': model.b}

with open('data/model.pkl', 'wb') as f:
    pickle.dump(dict_ab, f)

AttributeError: 'dict' object has no attribute 'a'

In [ ]:
!python data/hw1_check_ol.py data/model.pkl

('secret code 2:', '3c508')


In [ ]:
from nyoka import skl_to_pmml
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.datasets import load_diabetes

X, y = load_diabetes(return_X_y=True)
cols = load_diabetes()['feature_names']

scaler = MinMaxScaler()
pipe = Pipeline([  
            ('Scaling', MinMaxScaler()),
            ('Linear', LinearRegression())
        ])
# Обучение пайплайна, включающего линейную модель и нормализацию признаков
pipe.fit(X, y)
# Сохраним пайплайн в формате pmml в файл pipeline.pmml
skl_to_pmml(pipeline=pipe, col_names=cols, pmml_f_name="pipeline.pmml")

In [ ]:
with open('pipeline.pmml', 'r') as f:
    print(f.read())

<?xml version="1.0" encoding="UTF-8"?>
<PMML xmlns="http://www.dmg.org/PMML-4_4" version="4.4.1">
    <Header copyright="Copyright (c) 2021 Software AG" description="Default description">
        <Application name="Nyoka" version="5.5.0"/>
        <Timestamp>2025-12-17 13:24:18.311539</Timestamp>
    </Header>
    <DataDictionary numberOfFields="11">
        <DataField name="age" optype="continuous" dataType="double"/>
        <DataField name="sex" optype="continuous" dataType="double"/>
        <DataField name="bmi" optype="continuous" dataType="double"/>
        <DataField name="bp" optype="continuous" dataType="double"/>
        <DataField name="s1" optype="continuous" dataType="double"/>
        <DataField name="s2" optype="continuous" dataType="double"/>
        <DataField name="s3" optype="continuous" dataType="double"/>
        <DataField name="s4" optype="continuous" dataType="double"/>
        <DataField name="s5" optype="continuous" dataType="double"/>
        <DataField name

In [ ]:
import onnxruntime as rt 
from sklearn.datasets import load_boston
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from skl2onnx import to_onnx  # или convert_sklearn — оба работают, но в примерах чаще to_onnx
from skl2onnx.common.data_types import FloatTensorType  # тип для float-входа


# загружаем данные
X, y = load_boston(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=7)
print(X_train.shape, X_test.shape)

# обучаем модель
model = LinearRegression()
model.fit(X_train, y_train)  # обучаем на тренировочных данных

# делаем инференс моделью на тесте
test_pred = model.predict(X_test)  # предсказание на тестовых данных
print('sklearn model predict:\n', test_pred)

# конвертируем модель в ONNX-формат
initial_type = [('float_input',FloatTensorType([None, X_train.shape[1] ]))]  # None — batch size, второе измерение — число фичей (13 для Boston)
model_onnx = to_onnx(model, initial_types=initial_type)  # основная функция конвертации (альтернатива: convert_sklearn)

# сохраняем модель в файл
with open("model.onnx", "wb") as f:
	f.write(model_onnx.SerializeToString())
 	 
# Делаем инференс на тесте через ONNX-runtime
sess = rt.InferenceSession("model.onnx")  # создание сессии инференса
input_name = sess.get_inputs()[0].name
label_name = sess.get_outputs()[0].name
test_pred_onnx = sess.run([label_name],
                	{input_name:  X_test.astype(np.float32)})[0].reshape(-1)
print('onnx model predict:\n',test_pred_onnx)